# Übung 09: Statistik-Grundlagen

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 9 (Statistik-Grundlagen), ergänzend 7.3** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | Skalenniveaus, Lage-/Streuungs-/Formmasse von Hand, Normalverteilung & empirische Regel, fehlende Werte (MCAR/MAR/MNAR), Korrelation & Simpson-Paradox, Stichprobe & Verzerrung, p-Wert, Konfidenzintervall, t-Test/Mann-Whitney, Chi²-Test |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 240 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Sie arbeiten in der Analytics-Abteilung eines fiktiven regionalen Detailhändlers. Auf Ihrem Tisch liegen: eine Haushaltsbefragung, ein A/B-Test des Webshops, Kassenbons zweier Filialen und ein Lehrbuch-Datensatz.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

Dieses Notebook ist stärker auf **Interpretation** ausgerichtet – genau das wird in Teil A der Prüfung verlangt. Rechnen Sie, wo verlangt, zuerst **von Hand** (Rechenweg aufschreiben!) und prüfen Sie danach mit Code.

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify) -> folder 'data_uebung09'
# ---------------------------------------------------------------
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(909)
D = 'data_uebung09'
os.makedirs(D, exist_ok=True)

# (1) Household survey: income is missing more often for high incomes (MNAR)
n = 1200
age = np.clip(np.round(rng.normal(47, 14, n)), 18, 90).astype(int)
region = rng.choice(['Stadt', 'Agglomeration', 'Land'], size=n, p=[0.45, 0.35, 0.20])
hh = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.3, 0.33, 0.15, 0.15, 0.07])
log_inc = 8.85 + 0.012 * (age - 47) - 0.0003 * (age - 47) ** 2 + np.where(region == 'Stadt', 0.06, 0) + rng.normal(0, 0.42, n)
income_full = np.round(np.exp(log_inc), -1)
owner = (rng.random(n) < 1 / (1 + np.exp(-(log_inc - 9.0) * 2.2))).astype(int)
p_miss = 1 / (1 + np.exp(-(log_inc - 9.35) * 3.2))
income = np.where(rng.random(n) < p_miss, np.nan, income_full)
sat = np.clip(np.round(6.5 + 1.2 * (log_inc - 8.85) + rng.normal(0, 1.6, n)), 1, 10).astype(int)
pd.DataFrame({'respondent_id': np.arange(1, n + 1), 'age': age, 'region': region, 'household_size': hh,
              'home_owner': owner, 'income_chf': income, 'satisfaction': sat}).to_csv(f'{D}/survey.csv', index=False)
true_median = float(np.median(income_full))

# (2) A/B test of a web shop: variant B was shown more often on desktop (allocation bug)
m = 8000
variant = rng.choice(['A', 'B'], size=m)
desktop_p = np.where(variant == 'A', 0.30, 0.70)
device = np.where(rng.random(m) < desktop_p, 'desktop', 'mobile')
conv_p = np.select([(variant == 'A') & (device == 'desktop'), (variant == 'B') & (device == 'desktop'),
                    (variant == 'A') & (device == 'mobile')], [0.092, 0.080, 0.030], default=0.024)
converted = (rng.random(m) < conv_p).astype(int)
pd.DataFrame({'user_id': np.arange(1, m + 1), 'variant': variant, 'device': device,
              'converted': converted}).to_csv(f'{D}/abtest.csv', index=False)

# (3) Basket values of two independent shops (400 receipts each)
a = np.round(np.exp(rng.normal(3.45, 0.55, 400)), 2)
b = np.round(np.exp(rng.normal(3.30, 0.60, 400)), 2)
b[rng.choice(400, 4, replace=False)] = np.round(rng.uniform(600, 900, 4), 2)   # a few very large orders
pd.DataFrame({'shop': ['Altstadt'] * 400 + ['Seen'] * 400, 'basket_chf': np.concatenate([a, b])}).to_csv(f'{D}/baskets.csv', index=False)

# (4) Anscombe's quartet (Anscombe 1973)
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
quartet = pd.DataFrame({
    'set': np.repeat(['I', 'II', 'III', 'IV'], 11),
    'x': x123 * 3 + [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
    'y': [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68,
          9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74,
          7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73,
          6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]})
quartet.to_csv(f'{D}/quartet.csv', index=False)

print('Files created:', sorted(os.listdir(D)))
print("Referenzwert Steueramt: Median-Haushaltseinkommen der befragten Region =", f"{true_median:,.0f}".replace(',', "'"), 'CHF/Monat')

### Daten laden

| Datei | Inhalt |
|---|---|
| `survey.csv` | Haushaltsbefragung: `age`, `region`, `household_size`, `home_owner` (0/1), `income_chf` (Monatseinkommen, freiwillig), `satisfaction` (1–10) |
| `abtest.csv` | A/B-Test Webshop: `variant` (A/B), `device` (desktop/mobile), `converted` (0/1 = Kauf) |
| `baskets.csv` | Kassenbons zweier **unabhängiger** Filialen: `shop`, `basket_chf` |
| `quartet.csv` | Anscombe-Quartett: vier Datensätze `set` mit `x`, `y` |

In [ ]:
# Load the data (do not modify)
df_survey = pd.read_csv('data_uebung09/survey.csv')
df_ab = pd.read_csv('data_uebung09/abtest.csv')
df_baskets = pd.read_csv('data_uebung09/baskets.csv')
df_quartet = pd.read_csv('data_uebung09/quartet.csv')
for name, d in [('survey', df_survey), ('abtest', df_ab), ('baskets', df_baskets), ('quartet', df_quartet)]:
    print(f'{name:<8}', d.shape)

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 60 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (16 Punkte)

*Antworten ohne Bezug zu den Daten bzw. ohne Rechenweg werden nicht gewertet.*

### A1 (4 Punkte)

Bestimmen Sie das **Skalenniveau** (nominal, ordinal, intervall, verhältnis) und ob die Variable **diskret oder stetig** ist: `region`, `household_size`, `home_owner`, `income_chf`, `satisfaction`, `age`, `variant`, `basket_chf`.

Was bedeutet der Mittelwert von `home_owner`? Darf man für `satisfaction` einen Mittelwert berechnen – was spricht dafür, was dagegen? *(→ 9.1)*

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Sieben Kassenbons (CHF): `12, 15, 15, 18, 21, 24, 75`

Berechnen Sie **von Hand** (Rechenweg!): Mittelwert, Median, Modus, Spannweite, Stichproben-Standardabweichung `s` und den z-Wert des Bons über 75 CHF. Prüfen Sie danach mit pandas (inkl. IQR mit `quantile`). Wie verändern sich Mittelwert und Median, wenn der Bon über 75 CHF fehlt? Was folgt daraus für die Wahl des Lagemasses? *(→ 9.2)*

**Rechenweg und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code zum Überprüfen hier eintragen

### A3 (4 Punkte)

Ein t-Test vergleicht die mittleren Bonbeträge zweier Filialen und liefert `p = 0.03`. Beurteilen Sie jede Aussage als **richtig oder falsch** und begründen Sie in einem Satz: *(→ 9.6)*

1. «Mit 97 % Wahrscheinlichkeit unterscheiden sich die Filialen.»
2. «Wenn es in Wahrheit keinen Unterschied gäbe, wäre ein mindestens so grosser beobachteter Unterschied in etwa 3 % der Stichproben zu erwarten.»
3. «Der Unterschied ist gross und deshalb geschäftlich relevant.»
4. «Bei α = 0.05 verwerfen wir die Nullhypothese.»
5. «Wäre p = 0.30, wäre bewiesen, dass es keinen Unterschied gibt.»

**Ihre Antwort:**

> [Hier eintragen]

### A4 (4 Punkte)

Der Webshop hat 400'000 Besuche ausgewertet: Mit dem neuen Checkout steigt der durchschnittliche Warenkorb von CHF 84.10 auf CHF 84.40 (`p < 0.001`). Das Projekt kostete CHF 250'000.

Ist das Ergebnis «signifikant», ist es «relevant»? Erklären Sie den Unterschied mit diesen Zahlen und nennen Sie, welche **zusätzliche** Angabe zur Unsicherheit Sie im Bericht erwarten. *(→ 9.6)*

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (24 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber eine falsche oder irreführende Schlussfolgerung.*

### B1 (8 Punkte)

**Ziel:** Das typische Haushaltseinkommen der befragten Region schätzen. Das **Steueramt** weist für dieselbe Region einen Median aus (Ausgabe im Setup).

In [ ]:
# Original code (do not modify this cell)
df_B1_orig = df_survey.dropna()
print('Auswertbare Antworten:', len(df_B1_orig))
print('Durchschnittseinkommen (CHF):', round(df_B1_orig['income_chf'].mean()))
print('Fazit: Die Befragung trifft den Wert des Steueramts fast genau – sie ist also repräsentativ.')

**Aufgabe:** Untersuchen Sie das Fehlen der Einkommensangabe systematisch: Anteil fehlend, Vergleich der Gruppen **mit vs. ohne** Angabe bezüglich `age`, `region`, `household_size`, `home_owner` und `satisfaction` (z.B. `groupby(income_chf.isna())`). Welcher Mechanismus ist am plausibelsten und warum? Schreiben Sie einen korrigierten Bericht-Code (robustes Lagemass, Anteil fehlender Werte, Richtung der Verzerrung als Text) und begründen Sie, warum eine Imputation mit dem Median der Antwortenden das Problem **nicht** löst. *(→ 6.4, 9.4)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code hier eintragen

### B2 (8 Punkte)

**Ziel:** Entscheiden, welche Variante (A oder B) des Webshops besser konvertiert.

In [ ]:
# Original code (do not modify this cell)
rate = df_ab.groupby('variant')['converted'].mean() * 100
print(rate.round(2))
print('Fazit: Variante B konvertiert besser und soll für alle Nutzer eingeführt werden.')

**Aufgabe:** Analysieren Sie die Konversionsrate zusätzlich **getrennt nach Gerät** (`device`) und prüfen Sie, wie die Geräte auf die Varianten verteilt sind (Kreuztabelle mit Zeilenanteilen). Erklären Sie, warum das Gesamtergebnis in die Irre führt (Fachbegriff!), und schreiben Sie den korrigierten Code mit einer Tabelle Konversionsrate **Variante × Gerät** sowie dem korrigierten Fazit. Was hätte man beim Versuchsdesign anders machen müssen? *(→ 9.5, 9.6)*

**Begründung der Änderungen und korrigiertes Fazit:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code hier eintragen

### B3 (8 Punkte)

**Ziel:** Prüfen, ob sich die Bonbeträge der **unabhängigen** Filialen *Altstadt* und *Seen* unterscheiden.

In [ ]:
# Original code (do not modify this cell)
a = df_baskets.loc[df_baskets['shop'] == 'Altstadt', 'basket_chf']
b = df_baskets.loc[df_baskets['shop'] == 'Seen', 'basket_chf']

t, p = stats.ttest_rel(a, b)
print(f'Mittelwerte: Altstadt {a.mean():.2f} | Seen {b.mean():.2f}')
print(f't = {t:.2f}, p = {p:.3f}')

**Aufgabe:** Erklären Sie, warum `ttest_rel` hier methodisch falsch ist, obwohl er ohne Fehlermeldung läuft (Tipp: Was wird bei einem gepaarten Test paarweise verglichen?). Untersuchen Sie die Verteilungen kurz (Schiefe, Ausreisser, Varianzen) und schreiben Sie den korrigierten Code mit **zwei** passenden Tests: Welch-t-Test und Mann-Whitney-U-Test. Berichten Sie zusätzlich Median und Mittelwert pro Filiale und interpretieren Sie das Ergebnis. Begründen Sie jede Änderung. *(→ 9.6)*

**Begründung der Änderungen und Interpretation:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code hier eintragen

---
## Teil C – Python-Code schreiben (20 Punkte)

*Schreiben Sie Python-Code und interpretieren Sie jeweils in 1–2 Sätzen.*

### C1 (4 Punkte)

Simulieren Sie die drei Fehlwert-Mechanismen auf **vollständigen** Daten, um ihre Wirkung zu sehen. Verwenden Sie:

```python
rng = np.random.default_rng(1)
age = rng.integers(20, 80, 5000)
income = np.exp(8.5 + 0.01 * age + rng.normal(0, 0.4, 5000))
```

Erzeugen Sie drei Kopien von `income`, in denen jeweils ca. 30 % der Werte fehlen: **MCAR** (rein zufällig), **MAR** (Fehlwahrscheinlichkeit steigt mit `age`) und **MNAR** (Fehlwahrscheinlichkeit steigt mit `income` selbst). Vergleichen Sie in einer Tabelle wahren Mittelwert, Mittelwert nach `dropna()` und die Verzerrung in %. Zeigen Sie für MAR zusätzlich, dass eine **gruppenweise** Imputation (Median pro Altersklasse) die Verzerrung verringert. *(→ 9.4, 6.4)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C2 (4 Punkte)

Berechnen Sie für die Bonbeträge der Filiale *Altstadt*:

1. ein 95 %-Konfidenzintervall für den **Mittelwert** mit der t-Verteilung (`stats.t.interval(0.95, df=n-1, loc=mean, scale=stats.sem(x))`),
2. ein 95 %-**Bootstrap**-Konfidenzintervall für den **Median** (2'000 Wiederholungen, `rng = np.random.default_rng(42)`, Ziehen mit Zurücklegen, 2.5 %- und 97.5 %-Quantil der Bootstrap-Mediane).

Stellen Sie die Bootstrap-Verteilung als Histogramm mit den Intervallgrenzen dar. Interpretieren Sie das Intervall korrekt (was bedeuten «95 %»?). *(→ 9.6)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

Prüfen Sie die **empirische Regel (68–95–99.7)** für `age` und für `income_chf` (ohne fehlende Werte) aus der Befragung: Berechnen Sie für beide den Anteil der Werte innerhalb von ±1, ±2 und ±3 Standardabweichungen um den Mittelwert und stellen Sie die Ergebnisse den theoretischen Werten in einer Tabelle gegenüber. Ergänzen Sie je einen QQ-Plot. Für welche Variable ist die Regel brauchbar und warum? *(→ 9.3, 8.5)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C4 (4 Punkte)

Testen Sie **nur für Desktop-Nutzer** mit einem Chi²-Unabhängigkeitstest, ob `variant` und `converted` zusammenhängen. Geben Sie die beobachtete Kreuztabelle, die **erwarteten** Häufigkeiten (`expected` aus `stats.chi2_contingency`), Chi², Freiheitsgrade und den p-Wert aus. Was sagen die erwarteten Häufigkeiten aus, und wie lautet Ihr Fazit für die Desktop-Nutzer? Was bedeutet ein nicht signifikantes Ergebnis hier – und was nicht? *(→ 9.6)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C5 (4 Punkte)

Berechnen Sie für jeden der vier Datensätze in `quartet.csv` Mittelwert und Standardabweichung von `x` und `y`, die Pearson- und die Spearman-Korrelation (je 2 Stellen) in **einer** Tabelle. Erstellen Sie dann eine 2×2-Grafik mit je einem Scatterplot inkl. Regressionsgerade (`sns.regplot`).

Was ist die zentrale Lektion dieses Datensatzes? Beschreiben Sie für jeden der vier Datensätze in einem Halbsatz, warum Pearson-r allein in die Irre führt bzw. angemessen ist. *(→ 7.1, 9.5)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')